# Light Text Cleaning Only

**Scope:** This notebook handles ONLY the shared preprocessing that both
downstream teams (Sentiment + Topic Modelling) need. It does NOT tokenize,
lemmatize, remove stopwords, or create n-grams — those are for later steps before model creation

**Input:** Folder of MDA `.txt` files named like `Alphabet Inc._10-Q_2025-09-01_MDA.txt`

**Output:** Single DataFrame `df_shared` with:
- `doc_id`, `company_name`, `filing_type`, `filing_date`, `year`, `quarter`
- `sentences` — list of clean sentences (→ Sentiment Analysis team)
- `clean_text` — full cleaned text rejoined (→ Topic Analysis team)
- `n_sentences` — sentence count for QA

**Pipeline Steps:**
1. **Section-level removal** — FLS / safe harbor, tables, headers, boilerplate paragraphs
2. **Text cleaning** — lowercase, strip `$/%`, normalize whitespace
3. **Sentence segmentation** — split + drop fragments < 10 words

## Imports & Config

In [6]:
import re
import os
import pandas as pd
import nltk
from pathlib import Path
from nltk.tokenize import sent_tokenize

nltk.download('punkt_tab', quiet=True)

# ── Config ──────────────────────────────────────────────────────────
MDA_FOLDER       = "../MDA"  
FILE_PATTERN     = "*_MDA.txt"    
MIN_SENT_WORDS   = 10              # drop sentences shorter than this

## Step 1 — Section-Level Removal Patterns

Three categories of content to remove before any text analysis:
1. **Forward-looking statements (FLS)** — safe harbor boilerplate at the start
2. **Tables & numeric blocks** — financial statement tables, data rows, orphan numbers
3. **Boilerplate paragraphs** — accounting policy definitions, off-balance sheet disclaimers, etc.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1A. FORWARD-LOOKING STATEMENT BOUNDARY
# ═══════════════════════════════════════════════════════════════════
FLS_END_ANCHORS = [
    # Company / business overview variants
    r"Overview\s+Our\s+Company\s+and\s+Our\s+Businesses",
    r"Overview\s+Our\s+Company",
    r"Business\s+Overview\s",
    r"Company\s+Overview\s",
    r"Corporate\s+Overview\s",
    r"General\s+Overview\s",
    r"Overview\s+of\s+the\s+Company\s",
    r"Overview\s+of\s+Our\s+Business\s",
    r"Overview\s",
    r"(?:Overview|OVERVIEW)\s", 

    # Executive summary
    r"Executive\s+Summary\s",
    r"Executive\s+Overview\s",
    r"Management\s+Summary\s",
]

_FLS_PATTERN = re.compile("|".join(FLS_END_ANCHORS), re.IGNORECASE)


# ═══════════════════════════════════════════════════════════════════
# 1B. TABLE & NUMERIC BLOCK REMOVAL
# ═══════════════════════════════════════════════════════════════════

_VAL = (
    r'(?:\$\s*)?-?[\d,]+(?:\.\d+)?\s*(?:%|pts)?'
    r'|\([0-9,\.]+\)\s*(?:%|pts)?'
)
_DATE = (
    r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    r'\s+\d{1,2},?\s*\d{4}'
)
_PERIOD = (
    r'Three\s+Months\s+Ended'
    r'|Six\s+Months\s+Ended'
    r'|Nine\s+Months\s+Ended'
    r'|Twelve\s+Months\s+Ended'
    r'|Year\s+Ended'
    r'|Fiscal\s+Year\s+Ended'
    r'|Quarter-over-Quarter\s+Change'
    r'|Year-over-Year\s+Change'
    r'|\$\s*Change'
    r'|%\s*Change'
)
_UNITS = r'\(\$?\s*[Ii]n\s+(?:thousand|million|billion)[^)]*\)'

RE_HEADER_ROW = re.compile(
    r'(?:{date}|{period}|{units})'
    r'(?:\s+(?:{date}|{period}|{units}))*'
    .format(date=_DATE, period=_PERIOD, units=_UNITS),
    re.IGNORECASE
)
RE_DATA_ROW = re.compile(
    r'(?<![.!?]\s)'
    r'[A-Za-z][A-Za-z0-9 \&\(\),\-\.]{{1,50}}'
    r'(?:\s+(?:{val})){{2,}}'
    .format(val=_VAL),
    re.IGNORECASE
)
RE_FOLLOWING_TABLE = re.compile(r'[Tt]he following table[^.]+\.', re.IGNORECASE)
RE_ORPHAN_NUMS = re.compile(
    r'(?<!\w)(?:{val})(?:\s+(?:{val})){{0,3}}(?!\s*\w{{4}})'.format(val=_VAL),
    re.IGNORECASE
)
RE_PAGE_NUM = re.compile(r'\b\d{1,3}\b(?=\s+[A-Z])')

RE_TABLE_INTRO = re.compile(
    r'(?:[Tt]?he\s+following\s+(?:table|chart|graph|diagram|schedule|summary)'
    r'|[Tt]?he\s+table\s+below'
    r'|set\s+forth\s+below\s+is'
    r'|below\s+is\s+a\s+summary'
    r'|[Tt]?he\s+following\s+(?:is\s+a\s+)?(?:reconciliation|summary|illustrat)'
    r'|(?:a\s+summary|[Tt]?he\s+components?)\s+(?:of\s+\w+\s+)?(?:is|was|were|are)\s+as\s+follows'
    r'|(?:revenue|sales|expenses?|income|results|costs?)\s+(?:by\s+\w+\s+)?(?:is|was|were|are)\s+as\s+follows'
    r')[^.:]*[.:]'
, re.IGNORECASE)

RE_TABLE_LABELS = re.compile(
    r'\((?:In\s+)?(?:thousands|millions|billions)(?:\s*,\s*except[^)]+)?\)'
    r'\((?:In\s+)?(?:thousand|million|billion)(?:\s*,\s*except[^)]+)?\)'
    r'|\((?:Unaudited|Audited)\)'
    r'|\(As\s+a\s+Percentage\s+of[^)]+\)'
, re.IGNORECASE)

RE_COLUMN_JUNK = re.compile(
    r'(?:[\s\$%\(\)\-,]|Variance|Amount|Change|Total){3,}',
    re.IGNORECASE
)

# ═══════════════════════════════════════════════════════════════════
# 1C. BOILERPLATE PARAGRAPH REMOVAL
# ═══════════════════════════════════════════════════════════════════
S = r'[\s,/&]+'

BOILERPLATE_SENTENCE_PATTERNS = [

    # ═══════════════════════════════════════════════════════════════════
    # CATEGORY 1: TEMPORAL/PERIOD MARKERS
    # ═══════════════════════════════════════════════════════════════════
    # Patterns for fiscal periods, fiscal years, and temporal references
    
    re.compile(rf'(?:fiscal|calendar){S}(?:year|period){S}(?:ended?|ending)', re.I),
    re.compile(rf'(?:first|second|third|fourth){S}(?:quarter|year|half|nine\s+months)', re.I),
    re.compile(rf'(?:during|for){S}(?:the|a){S}(?:first|second|third|fourth){S}(?:quarter|year)', re.I),
    re.compile(rf'nine{S}months{S}ended?', re.I),
    re.compile(rf'(?:first|second){S}half{S}(?:of|the)', re.I),
    re.compile(rf'fiscal{S}(?:year|period){S}(?:20\d{2}|ended)', re.I),
    re.compile(rf'(?:year|first_year|second_year|third_year|fourth_year){S}to{S}date', re.I),
    re.compile(rf'(?:current|prior){S}(?:fiscal|calendar){S}(?:year|period)', re.I),
    re.compile(rf'(?:same|corresponding){S}(?:period|month|year){S}(?:of|in)?{S}(?:the|prior)', re.I),

    # ═══════════════════════════════════════════════════════════════════
    # CATEGORY 2: BOILERPLATE CROSS-REFERENCE LANGUAGE
    # ═══════════════════════════════════════════════════════════════════
    # Original cross-reference patterns
    
    re.compile(rf'(?:please{S})?(?:see|refer){S}(?:to{S})?note[s]?{S}(?:\d+{S})?(?:to{S})?(?:the{S})?(?:condensed{S})?(?:consolidated{S})?financial', re.I),
    re.compile(rf'see{S}accompanying{S}notes', re.I),
    re.compile(rf'read{S}(?:in{S})?conjunction{S}with', re.I),
    re.compile(rf'(?:included|appearing){S}(?:elsewhere{S})?in{S}(?:this|the|our){S}(?:quarterly|annual){S}report', re.I),
    re.compile(rf'(?:form|filed){S}(?:on{S})?(?:form{S})?10-[kq]', re.I),
    
    # New patterns for cross-reference boilerplate
    re.compile(rf'(?:please|note|refer){S}to{S}(?:the|our)?{S}(?:discussion|additional(?:{S}information)?)', re.I),
    re.compile(rf'(?:for|in){S}(?:further|additional){S}(?:discussion|information|details?)', re.I),
    re.compile(rf'(?:see|refer|please{S}see){S}(?:the|our){S}(?:discussion|section){S}(?:entitled|titled)', re.I),
    re.compile(rf'(?:for{S})?(?:recently{S})?(?:issued|adopted){S}accounting{S}(?:standards?|pronouncements?)', re.I),
    re.compile(rf'(?:discussed|discussed{S}in|described{S}in){S}(?:note|section)', re.I),
    re.compile(rf'additional{S}(?:discussion|information|details?){S}(?:can|may){S}be{S}found', re.I),

    # ═══════════════════════════════════════════════════════════════════
    # CATEGORY 3: GENERIC QUANTITATIVE TERMS
    # ═══════════════════════════════════════════════════════════════════
    re.compile(r'\b(?:a[\s\.\,]*)?table[\s\.\,]*of[\s\.\,]*contents\b', re.I), # table of contents
    re.compile(rf'(?:in{S})?(?:millions?|billions?|thousands?){S}(?:of|dollars?)?', re.I),
    re.compile(rf'(?:approximately|roughly){S}(?:$|\d)', re.I),
    re.compile(rf'(?:total|net|gross){S}(?:revenue|sales|expenses?|amount)', re.I),
    re.compile(rf'(?:respectively|in{S}the{S}same|comparable){S}(?:period|amount)', re.I),
    re.compile(rf'(?:dollar|amount){S}(?:amounts?|in{S}thousands?|in{S}millions?)', re.I),
    re.compile(rf'(?:increased|decreased|improved|declined){S}(?:to|by)?{S}(?:approximately|\$|(?:\d+))', re.I),

    # ═══════════════════════════════════════════════════════════════════
    # CATEGORY 4: GENERIC FUNCTIONAL WORDS (Low Semantic Value)
    # ═══════════════════════════════════════════════════════════════════
    # Phrases with generic auxiliaries and modals that rarely provide meaningful content
    
    re.compile(rf'(?:may|could|would|should|might){S}(?:not{S})?(?:be|have){S}(?:affected|impacted|changed)', re.I),
    re.compile(rf'(?:due{S})?to{S}(?:certain|various|several){S}(?:factors?|reasons)', re.I),
    re.compile(rf'(?:based{S})?(?:on|upon){S}(?:our|the|current){S}(?:estimates?|assumptions?|judgments?)', re.I),
    re.compile(rf'(?:is|was|were){S}(?:also{S})?(?:new|newly|currently)', re.I),
    re.compile(rf'(?:using|used{S})?(?:new|updated){S}(?:methodology|method|approach)', re.I),
    re.compile(rf'(?:as{S})?(?:well{S})?as{S}(?:our|the)', re.I),
    re.compile(rf'(?:for{S})?us(?:{S}to|{S}(?:and|the))', re.I),
    re.compile(rf'(?:currently|presently){S}(?:does|is|are)', re.I),
    re.compile(rf'(?:could{S})?affect{S}(?:our|the){S}(?:results?|operations?)', re.I),
    re.compile(rf'in{S}(?:addition|order|part)', re.I),

    # ═══════════════════════════════════════════════════════════════════
    # CATEGORY 5: GEOGRAPHIC/JURISDICTION BOILERPLATE
    # ═══════════════════════════════════════════════════════════════════
    
    re.compile(rf'(?:united{S}states?|u\.?s\.?|outside{S}(?:the|our)?{S}(?:united{S})?states?)', re.I),
    re.compile(rf'(?:domestic|international|foreign|non-?us){S}(?:operations?|revenue|sales)', re.I),
    re.compile(rf'(?:in|outside|within){S}(?:the|our)?{S}(?:united{S}states?|u\.s\.)', re.I),
    re.compile(rf'(?:united{S}states?|foreign){S}operations?{S}(?:represent|comprise)', re.I),
    re.compile(rf'(?:foreign{S})?exchange{S}(?:rates?|fluctuations?|gains?|losses?)', re.I),
    re.compile(rf'(?:geographic|jurisdictional){S}mix{S}of{S}(?:revenues?|sales)', re.I),

    # ═══════════════════════════════════════════════════════════════════
    # CATEGORY 6: COMPANY METADATA (HQ, WEBSITE, CONTACT INFO)
    # ═══════════════════════════════════════════════════════════════════

    re.compile(rf'(?:headquarters?|head\s*office){S}(?:is|are|located|based)?{S}', re.I),
    re.compile(rf'(?:headquartered?|head\s*office){S}(?:in|are|located|based)?{S}', re.I),
    re.compile(rf'(?:located|based){S}(?:in|at){S}.*(?:headquarters?|head\s*office)', re.I),

    # Website / URL patterns
    re.compile(rf'(?:website|web\s*site){S}(?:is|at|located)?{S}', re.I),
    re.compile(r'https?://\S+', re.I),
    re.compile(r'www\.\S+', re.I),

    # Investor relations / contact pages
    re.compile(rf'investor{S}relations{S}(?:website|page|site)?', re.I),
    re.compile(rf'(?:visit|see){S}(?:our{S})?(?:website|web\s*site)', re.I),

    # Address-like patterns (optional but useful)
    re.compile(rf'\d+{S}.*(?:street|st\.?|avenue|ave\.?|road|rd\.?|blvd|drive|dr\.?)', re.I),
    re.compile(rf'(?:city|state|zip|postal){S}(?:code)?', re.I),



    # ── Forward-looking / safe harbor (catch stragglers after FLS cut) ──
    re.compile(rf'forward[_-]?looking{S}statements?', re.I),
    re.compile(rf'actual{S}results?{S}(?:could|may|might|will){S}(?:differ|be){S}materially', re.I),
    re.compile(rf'(?:should|do){S}not{S}place{S}undue{S}reliance', re.I),
    re.compile(rf'(?:no|assume{S}no){S}obligation{S}(?:to{S})?update', re.I),
    re.compile(rf'(?:known|unknown){S}risks?{S}uncertainties', re.I),
    re.compile(rf'safe{S}harbor', re.I),
    re.compile(rf'private{S}securities{S}litigation{S}reform', re.I),

    # ── Accounting policy boilerplate ──
    re.compile(rf'critical{S}accounting{S}polic(?:y|ies){S}(?:and{S})?estimates?', re.I),
    re.compile(rf'(?:significant|critical){S}accounting{S}polic(?:y|ies)', re.I),
    re.compile(rf'managements?{S}discussion{S}(?:and{S})?analysis{S}(?:of{S})?financial{S}condition', re.I),
    re.compile(rf'preparation{S}(?:of{S})?financial{S}(?:statements?{S})?.*requires{S}(?:us|management){S}(?:to{S})?make{S}estimates', re.I),
    re.compile(rf'actual{S}results{S}could{S}differ{S}from{S}(?:those{S})?estimates', re.I),
    re.compile(rf'generally{S}accepted{S}accounting{S}principles', re.I),
    re.compile(rf'(?:management|we){S}discussed{S}(?:the{S})?development{S}(?:and{S})?selection{S}(?:of{S})?critical', re.I),
    re.compile(rf'audit{S}committee{S}(?:has{S})?reviewed{S}(?:the{S})?disclosures', re.I),

    # ── Fair value hierarchy ──
    re.compile(rf'fair{S}value{S}hierarchy{S}prioriti[sz]es', re.I),
    re.compile(rf'level{S}[123]{S}(?:inputs|consists?|assets?|measurements?)', re.I),
    re.compile(rf'quoted{S}(?:market{S})?prices{S}.*active{S}markets?{S}(?:for{S})?identical', re.I),
    re.compile(rf'unobservable{S}inputs{S}(?:for{S}the{S}asset|that{S}are{S}supported)', re.I),

    # ── Tax boilerplate ──
    re.compile(rf'(?:full{S})?valuation{S}allowance{S}(?:against|on){S}.*deferred{S}tax', re.I),

    # ── Goodwill & impairment ──
    re.compile(rf'goodwill{S}(?:is{S})?subject{S}(?:to{S})?(?:an{S})?(?:annual{S})?impairment{S}test', re.I),
    re.compile(rf'other-?than-?temporar(?:y|ily){S}impair(?:ment|ed)', re.I),

    # ── Investment policy ──
    re.compile(rf'available-?for-?sale{S}(?:investments?|securities){S}(?:are{S})?subject{S}(?:to{S})?(?:a{S})?periodic{S}impairment', re.I),
    re.compile(rf'securities{S}(?:are{S})?(?:reported|carried|measured){S}(?:at{S})?fair{S}value', re.I),

    # ── Stock-based compensation ──
    re.compile(rf'black-?scholes{S}(?:option{S})?pricing{S}model', re.I),
    re.compile(rf'forfeitures{S}(?:are{S})?estimated{S}(?:based{S})?(?:on|upon){S}historical{S}experience', re.I),

    # ── Litigation ──
    re.compile(rf'(?:aggressively|vigorously){S}defending{S}.*litigation', re.I),
    re.compile(rf'legal{S}(?:proceedings|matters){S}(?:are{S})?inherently{S}unpredictable', re.I),
    re.compile(rf'(?:we{S})?record{S}(?:a{S})?(?:provision|liability).*(?:when{S})?.*(?:both{S})?probable', re.I),

    # ── Liquidity boilerplate ──
    re.compile(rf'(?:we{S})?believe{S}(?:that{S})?(?:our{S})?existing{S}(?:cash|liquidity).*(?:sufficient|adequate)', re.I),
    re.compile(rf'additional{S}(?:funds|financing){S}may{S}not{S}(?:be{S})?available{S}(?:on{S})?(?:favorable{S})?terms', re.I),
    re.compile(rf'(?:next{S})?(?:at{S}least{S})?(?:the{S})?next{S}(?:12|twelve){S}months', re.I),

    # ── Stock repurchase ──
    re.compile(rf'repurchases{S}(?:may{S}be{S})?made{S}(?:in{S}the{S})?open{S}market', re.I),
    re.compile(rf'(?:the{S})?program{S}(?:does{S})?not{S}obligate{S}.*common{S}stock', re.I),

    # ── Off-balance sheet ──
    re.compile(rf'(?:do|did){S}not{S}have{S}(?:any{S})?(?:material{S})?off-?balance{S}sheet', re.I),
    re.compile(rf'(?:structured{S}finance{S}or{S})?special{S}purpose{S}entities', re.I),
    re.compile(rf'unconsolidated{S}(?:entities|organizations)', re.I),

    # ── Contractual obligations ──
    re.compile(rf'(?:no{S})?material{S}changes{S}(?:in{S})?.*contractual{S}obligations', re.I),

    # ── Covenant compliance ──
    re.compile(rf'(?:was|were|is|are){S}in{S}compliance{S}with{S}all{S}(?:financial{S})?covenants', re.I),
    re.compile(rf'customary{S}(?:negative{S})?covenants{S}(?:that{S})?limit', re.I),

    # ── Geographic / segment disclosure ──
    re.compile(rf'revenue{S}(?:by{S})?geographic{S}(?:region|location){S}(?:is{S})?(?:based|allocated)', re.I),

    # ── Risk factors reference ──
    re.compile(rf'(?:discussed|described){S}(?:in{S})?(?:the{S})?(?:section{S})?(?:entitled|titled){S}risk{S}factors', re.I),

    # ── Table / numeric artifacts ──
    re.compile(r'^[\d\s,.$%()]+$'),
    re.compile(rf'^(?:dollars{S}(?:in{S})?(?:thousands|millions|billions))$', re.I),
    re.compile(rf'^(?:dollar{S}(?:in{S})?(?:thousand|million|billion))$', re.I),
    re.compile(rf'^(?:in{S}thousands{S}except{S}per{S}share)', re.I),
    re.compile(rf'^(?:unaudited)$', re.I),

    # ── Insurance-specific boilerplate ──
    re.compile(rf'(?:a{S})?combined{S}ratio{S}(?:of{S})?under{S}100', re.I),
    re.compile(rf'property{S}(?:and{S})?(?:/)?casualty{S}(?:insurance{S})?industry{S}(?:is{S})?cyclical', re.I),
]

# ── Structural noise patterns ──
RE_ITEM_TAG   = re.compile(r'\bItem\s+\d+[A-Z]?\.\s*')
RE_PART_TAG   = re.compile(r'\bPart\s+[IVX]+,?\s*')
RE_COPYRIGHT  = re.compile(r'©?\s*\d{4}\s+\w[^.]*(?:Corporation|Inc|LLC|Ltd)[^.]*\.')
RE_WHITESPACE = re.compile(r'\s{2,}')

## Helper Functions

In [8]:
def parse_filename(filename: str) -> dict:
    stem = Path(filename).stem
    m = re.match(
        r'^(?P<company>.+)_(?P<ftype>10-[KQ])_(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})(?:T[\d_]+)?_(?P<section>.+)',
        stem
    )
    if not m:
        return None

    company_name = m.group('company')
    filing_type  = m.group('ftype')
    year         = m.group('year')
    month        = int(m.group('month'))
    day          = m.group('day')
    quarter      = f"Q{(month - 1) // 3 + 1}"
    filing_date  = f"{year}-{m.group('month')}-{day}"

    return {
        "company_name": company_name,
        "filing_type":  filing_type,
        "filing_date":  filing_date,
        "year":         year,
        "quarter":      quarter,
    }

def strip_fls(raw: str, filename: str = "") -> str:
    """Step 1A: Remove everything before the first substantive section heading."""
    match = _FLS_PATTERN.search(raw)
    if match:
        return raw[match.start():]
    else:
        print(f"  FLS anchor not found in '{filename}' — using full doc.")
        return raw


def remove_tables(text: str) -> str:
    """Step 1B: Strip financial tables, data rows, orphan numbers."""
    text = RE_TABLE_INTRO.sub(' ', text)    
    text = RE_HEADER_ROW.sub(' ', text)
    text = RE_DATA_ROW.sub(' ', text)
    text = RE_TABLE_LABELS.sub(' ', text)   
    text = RE_ORPHAN_NUMS.sub(' ', text)
    text = RE_COLUMN_JUNK.sub(' ', text)    
    text = RE_PAGE_NUM.sub(' ', text)
    return text


def is_boilerplate_sentence(sentence: str) -> bool:
    """Step 1C: Check if a sentence matches any boilerplate pattern."""
    for pattern in BOILERPLATE_SENTENCE_PATTERNS:
        if pattern.search(sentence):
            return True
    return False


def clean_text(text: str) -> str:
    """
    Step 2: Basic text cleaning.
    - Strip structural tags (Item X, Part IV, copyright lines)
    - Lowercase
    - Remove standalone dollar amounts, percentages, numbers
    - Normalize whitespace
    - Preserve sentence-ending punctuation (. ! ?)
    """
    # Structural noise
    text = RE_ITEM_TAG.sub(' ', text)
    text = RE_PART_TAG.sub(' ', text)
    text = RE_COPYRIGHT.sub(' ', text)

    # Lowercase
    text = text.lower()

    # Strip $, %, standalone numbers (but keep words with embedded digits like "q3")
    text = re.sub(r'\$[\d,\.]+', ' ', text)           # dollar amounts
    text = re.sub(r'(?<!\w)[\d,\.]+%', ' ', text)     # percentages
    text = re.sub(r'(?<!\w)\d[\d,\.]*(?!\w)', ' ', text)  # standalone numbers

    # Light punctuation: keep . ! ? for sentence splitting, remove others
    text = re.sub(r'[^\w\s\.\!\?\-\']', ' ', text)

    # Normalize whitespace
    text = RE_WHITESPACE.sub(' ', text)
    return text.strip()


def segment_sentences(text: str) -> list:
    """
    Step 3: Split into sentences, drop fragments, remove boilerplate sentences.
    """
    raw_sents = sent_tokenize(text)
    clean_sents = []
    for s in raw_sents:
        s = s.strip()
        if len(s.split()) < MIN_SENT_WORDS:
            continue
        if is_boilerplate_sentence(s):
            continue
        clean_sents.append(s)
    return clean_sents

## Main Pipeline
1. **Section-level removal** — FLS / safe harbor, tables, headers, boilerplate paragraphs
2. **Text cleaning** — lowercase, strip `$/%`, normalize whitespace
3. **Sentence segmentation** — split + drop fragments < 10 words

In [9]:
def run_preprocessing(mda_folder: str, file_pattern: str) -> pd.DataFrame:
    """
    Shared preprocessing pipeline (Steps 1–3).

    Returns:
        df_shared — one row per filing with:
            sentences   (list[str])  → for Sentiment team
            clean_text  (str)        → for Topic team
    """
    mda_path = Path(mda_folder)
    files = sorted(mda_path.glob(file_pattern))
    print(f"Found {len(files)} MDA files in {mda_folder}")

    if not files:
        raise FileNotFoundError(f"No files matching '{file_pattern}' in {mda_folder}")

    rows = []
    skipped_meta = 0
    skipped_empty = 0

    for fp in files:
        # ── Parse metadata from filename ──
        meta = parse_filename(fp.name)
        if meta is None:
            print(f"  SKIPPED (bad filename): {fp.name}")
            skipped_meta += 1
            continue

        # ── Read raw text ──
        raw_text = fp.read_text(encoding='utf-8', errors='replace')

        # ── STEP 1A: Remove forward-looking statements ──
        text = strip_fls(raw_text, fp.name)

        # ── STEP 1B: Remove tables & numeric blocks ──
        text = remove_tables(text)

        # ── STEP 2: Text cleaning ──
        text = clean_text(text)

        # ── STEP 3: Sentence segmentation + boilerplate removal ──
        sentences = segment_sentences(text)

        if not sentences:
            print(f"  SKIPPED (no sentences after cleaning): {fp.name}")
            skipped_empty += 1
            continue

        # ── Build row ──
        doc_id = f"{meta['company_name']}_{meta['filing_type']}_{meta['filing_date']}"

        rows.append({
            "doc_id":       doc_id,
            "company_name": meta["company_name"],
            "filing_type":  meta["filing_type"],
            "filing_date":  meta["filing_date"],
            "year":         meta["year"],
            "quarter":      meta["quarter"],
            "sentences":    sentences,                  # list[str] → Sentiment team
            "clean_text":   ' '.join(sentences),        # str → Topic team
            "n_sentences":  len(sentences),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(["company_name", "filing_date"]).reset_index(drop=True)

    print(f"\nDone!")
    print(f"  Processed:  {len(rows)} documents")
    print(f"  Skipped:    {skipped_meta} (bad filename) + {skipped_empty} (empty after cleaning)")
    print(f"  Output:     {df.shape[0]} rows × {df.shape[1]} columns")
    return df

## Run & Export

In [10]:
df_shared = run_preprocessing(MDA_FOLDER, FILE_PATTERN)
# df_shared = run_preprocessing("./sample", FILE_PATTERN)

# ── Quick inspection ───────────────────────────────────────────────
print("\n=== df_shared (first 5 rows) ===")
print(df_shared[["doc_id", "company_name", "filing_type", "filing_date",
                  "n_sentences"]].head(5).to_string(index=False))

# ── Coverage summary ───────────────────────────────────────────────
print(f"\n=== Summary ===")
print(f"Total documents:      {len(df_shared)}")
print(f"Total sentences:      {df_shared['n_sentences'].sum():,}")
print(f"Avg sentences/doc:    {df_shared['n_sentences'].mean():.1f}")
print(f"Companies:            {df_shared['company_name'].nunique()}")

print("\n=== Documents per company (top 10) ===")
print(df_shared.groupby("company_name")["doc_id"].count()
      .sort_values(ascending=False).head(10).to_string())

Found 18183 MDA files in ../MDA
  FLS anchor not found in '1-800-PetMeds_10-K_2023-05-23_MDA.txt' — using full doc.
  FLS anchor not found in '1-800-PetMeds_10-K_2024-06-14_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2017-02-28_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2018-03-14_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2019-02-28_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2020-02-26_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2021-03-05_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2022-03-01_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2023-03-16_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2024-08-13_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-K_2025-03-27_MDA.txt' — using full doc.
  FLS anchor not found in '3D_Systems_10-Q_2015-05-06_MDA.txt' — using 

## Sample: What the output looks like

In [11]:
print("=== Sample sentences (first doc, first 3 sentences) ===")
sample_sents = df_shared.iloc[0]["sentences"][:3]
for i, s in enumerate(sample_sents):
    print(f"  [{i}] {s[:120]}{'...' if len(s) > 120 else ''}")

print(f"\n=== Sample clean_text (first doc, first 300 chars) ===")
print(df_shared.iloc[0]["clean_text"][:300] + "...")

=== Sample sentences (first doc, first 3 sentences) ===
  [0] the companys common stock is traded on the nasdaq global select market under the symbol pets.
  [1] the company began selling pet medications and other pet health products in september .
  [2] presently the companys product line includes approximately of the most popular pet medications health products and suppl...

=== Sample clean_text (first doc, first 300 chars) ===
the companys common stock is traded on the nasdaq global select market under the symbol pets. the company began selling pet medications and other pet health products in september . presently the companys product line includes approximately of the most popular pet medications health products and supp...


## Export

In [12]:
# # Save as parquet (preserves list columns natively)
# df_shared.to_parquet("mda_shared_preprocessed.parquet", index=False)
# print("Saved: mda_shared_preprocessed.parquet")
import json
df_csv = df_shared.copy()
df_csv["sentences"] = df_csv["sentences"].apply(json.dumps)
# full doc 
df_csv.to_csv("mda_shared_preprocessed.csv", index=False)
# sample 
df_csv[df_csv['company_name']=="NVIDIA"].to_csv("nvidia_sample.csv", index=False)
print("Saved: mda_shared_preprocessed.csv")


Saved: mda_shared_preprocessed.csv
